# Statistical Testing for Model Comparison

**Interview Focus**: Significance testing, bootstrap vs parametric, comparing classifiers.

**Key Concepts**:
- Bootstrap confidence intervals: non-parametric, distribution-free
- Paired t-test: parametric test for comparing two models on same data
- McNemar's test: comparing classifiers on binary outcomes
- Permutation test: exact non-parametric test

**Math Notes**:
- Bootstrap: sample with replacement B times, compute statistic each time, take percentiles
- McNemar: tests if disagreement pattern is symmetric, $\chi^2 = (b-c)^2 / (b+c)$
- Permutation test: under H0, labels are exchangeable between models

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

## 1. The Problem: Is Model A Better Than Model B?

You train two models on the same data and evaluate on the same test set. Model A gets 85.2% accuracy, Model B gets 83.8%. Is A truly better, or is this just noise?

In [ ]:
# Simulate two classifiers on the same test set
np.random.seed(42)
n_test = 500

# True labels
y_true = np.random.binomial(1, 0.5, n_test)

# Model A: slightly better (85% base accuracy)
noise_a = np.random.rand(n_test)
y_pred_a = np.where(noise_a < 0.852, y_true, 1 - y_true)

# Model B: slightly worse (84% base accuracy)
noise_b = np.random.rand(n_test)
y_pred_b = np.where(noise_b < 0.838, y_true, 1 - y_true)

acc_a = np.mean(y_pred_a == y_true)
acc_b = np.mean(y_pred_b == y_true)

print(f"Model A accuracy: {acc_a:.4f}")
print(f"Model B accuracy: {acc_b:.4f}")
print(f"Difference:       {acc_a - acc_b:.4f}")
print(f"\nIs this difference statistically significant?")

## 2. Bootstrap Confidence Intervals

The bootstrap is the workhorse of ML evaluation:
1. Resample the test set with replacement (B times)
2. Compute the metric on each bootstrap sample
3. Take percentiles for the confidence interval

No assumptions about the distribution of the metric.

In [ ]:
def bootstrap_ci(metric_fn, y_true, y_pred, n_bootstrap=10000, ci=0.95, seed=42):
    """Compute bootstrap confidence interval for a metric.
    
    Args:
        metric_fn: function(y_true, y_pred) -> scalar
        n_bootstrap: number of bootstrap resamples
        ci: confidence level (e.g., 0.95 for 95% CI)
    
    Returns:
        point_estimate, (lower, upper), bootstrap_distribution
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    
    point_estimate = metric_fn(y_true, y_pred)
    
    bootstrap_scores = []
    for _ in range(n_bootstrap):
        # Sample with replacement
        indices = rng.randint(0, n, size=n)
        score = metric_fn(y_true[indices], y_pred[indices])
        bootstrap_scores.append(score)
    
    bootstrap_scores = np.array(bootstrap_scores)
    alpha = (1 - ci) / 2
    lower = np.percentile(bootstrap_scores, 100 * alpha)
    upper = np.percentile(bootstrap_scores, 100 * (1 - alpha))
    
    return point_estimate, (lower, upper), bootstrap_scores


def accuracy_metric(y_true, y_pred):
    return np.mean(y_true == y_pred)


# Bootstrap CI for each model
est_a, ci_a, dist_a = bootstrap_ci(accuracy_metric, y_true, y_pred_a)
est_b, ci_b, dist_b = bootstrap_ci(accuracy_metric, y_true, y_pred_b)

print(f"Model A: {est_a:.4f} (95% CI: [{ci_a[0]:.4f}, {ci_a[1]:.4f}])")
print(f"Model B: {est_b:.4f} (95% CI: [{ci_b[0]:.4f}, {ci_b[1]:.4f}])")
print(f"\nOverlapping CIs do not necessarily mean no significant difference.")
print(f"We need to bootstrap the DIFFERENCE directly.")

In [ ]:
def bootstrap_paired_difference(metric_fn, y_true, y_pred_a, y_pred_b,
                                 n_bootstrap=10000, ci=0.95, seed=42):
    """Bootstrap the difference in metrics between two models (paired).
    
    Paired = same test set, so we resample indices and compute both metrics
    on the same bootstrap sample.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    
    point_diff = metric_fn(y_true, y_pred_a) - metric_fn(y_true, y_pred_b)
    
    diffs = []
    for _ in range(n_bootstrap):
        indices = rng.randint(0, n, size=n)
        score_a = metric_fn(y_true[indices], y_pred_a[indices])
        score_b = metric_fn(y_true[indices], y_pred_b[indices])
        diffs.append(score_a - score_b)
    
    diffs = np.array(diffs)
    alpha = (1 - ci) / 2
    lower = np.percentile(diffs, 100 * alpha)
    upper = np.percentile(diffs, 100 * (1 - alpha))
    
    # p-value: proportion of bootstrap samples where diff <= 0
    p_value = np.mean(diffs <= 0)
    
    return point_diff, (lower, upper), diffs, p_value


diff, ci_diff, diff_dist, p_val = bootstrap_paired_difference(
    accuracy_metric, y_true, y_pred_a, y_pred_b)

print(f"Difference (A - B): {diff:.4f}")
print(f"95% CI: [{ci_diff[0]:.4f}, {ci_diff[1]:.4f}]")
print(f"Bootstrap p-value: {p_val:.4f}")
print(f"\nSignificant at alpha=0.05? {'Yes' if p_val < 0.05 else 'No'}")
if ci_diff[0] > 0:
    print("CI does not contain 0 -> A is significantly better than B.")
else:
    print("CI contains 0 -> difference may not be significant.")

In [ ]:
# Visualize bootstrap distributions
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Model A distribution
ax = axes[0]
ax.hist(dist_a, bins=50, alpha=0.7, color='#2196F3', edgecolor='white')
ax.axvline(est_a, color='red', linewidth=2, label=f'Point est: {est_a:.4f}')
ax.axvline(ci_a[0], color='red', linestyle='--', alpha=0.7)
ax.axvline(ci_a[1], color='red', linestyle='--', alpha=0.7, label=f'95% CI')
ax.set_xlabel('Accuracy')
ax.set_ylabel('Count')
ax.set_title('Model A Bootstrap Distribution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Model B distribution
ax = axes[1]
ax.hist(dist_b, bins=50, alpha=0.7, color='#FF9800', edgecolor='white')
ax.axvline(est_b, color='red', linewidth=2, label=f'Point est: {est_b:.4f}')
ax.axvline(ci_b[0], color='red', linestyle='--', alpha=0.7)
ax.axvline(ci_b[1], color='red', linestyle='--', alpha=0.7, label=f'95% CI')
ax.set_xlabel('Accuracy')
ax.set_title('Model B Bootstrap Distribution')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

# Difference distribution
ax = axes[2]
ax.hist(diff_dist, bins=50, alpha=0.7, color='#4CAF50', edgecolor='white')
ax.axvline(0, color='black', linewidth=2, linestyle='--', label='No difference')
ax.axvline(diff, color='red', linewidth=2, label=f'Observed: {diff:.4f}')
ax.axvline(ci_diff[0], color='red', linestyle='--', alpha=0.7)
ax.axvline(ci_diff[1], color='red', linestyle='--', alpha=0.7, label='95% CI')
ax.set_xlabel('Accuracy Difference (A - B)')
ax.set_title(f'Paired Difference (p={p_val:.3f})')
ax.legend(fontsize=9)
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 3. Paired t-Test

Classical parametric test. For each sample, compute the per-sample metric difference, then test if the mean difference is zero.

$$t = \frac{\bar{d}}{s_d / \sqrt{n}}$$

where d_i = metric_A(sample_i) - metric_B(sample_i).

Assumes: differences are approximately normally distributed.

In [ ]:
def paired_t_test(y_true, y_pred_a, y_pred_b):
    """Paired t-test for comparing two classifiers.
    
    Per-sample: correct_a(i) - correct_b(i) for each test sample.
    H0: mean difference = 0.
    """
    # Per-sample correctness: 1 if correct, 0 if wrong
    correct_a = (y_pred_a == y_true).astype(float)
    correct_b = (y_pred_b == y_true).astype(float)
    
    # Differences
    d = correct_a - correct_b  # +1, 0, or -1 per sample
    n = len(d)
    
    mean_d = np.mean(d)
    std_d = np.std(d, ddof=1)  # sample std
    se = std_d / np.sqrt(n)
    
    if se == 0:
        return {'t_stat': 0, 'p_value': 1.0, 'mean_diff': mean_d, 'se': se}
    
    t_stat = mean_d / se
    
    # Two-sided p-value using normal approximation (valid for large n)
    # For exact: use t-distribution with n-1 df
    # Here we use the standard normal approximation
    from math import erfc, sqrt
    p_value = erfc(abs(t_stat) / sqrt(2))  # two-sided
    
    return {
        't_stat': t_stat,
        'p_value': p_value,
        'mean_diff': mean_d,
        'se': se,
        'ci_lower': mean_d - 1.96 * se,
        'ci_upper': mean_d + 1.96 * se,
    }


result = paired_t_test(y_true, y_pred_a, y_pred_b)
print(f"Paired t-test:")
print(f"  Mean difference: {result['mean_diff']:.4f}")
print(f"  Standard error:  {result['se']:.4f}")
print(f"  t-statistic:     {result['t_stat']:.4f}")
print(f"  p-value:         {result['p_value']:.4f}")
print(f"  95% CI:          [{result['ci_lower']:.4f}, {result['ci_upper']:.4f}]")
print(f"\n  Significant at alpha=0.05? {'Yes' if result['p_value'] < 0.05 else 'No'}")

In [ ]:
# Visualize the per-sample differences
correct_a = (y_pred_a == y_true).astype(float)
correct_b = (y_pred_b == y_true).astype(float)
diffs_per_sample = correct_a - correct_b

# Count categories
both_right = np.sum((correct_a == 1) & (correct_b == 1))
both_wrong = np.sum((correct_a == 0) & (correct_b == 0))
a_right_b_wrong = np.sum((correct_a == 1) & (correct_b == 0))
a_wrong_b_right = np.sum((correct_a == 0) & (correct_b == 1))

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Disagreement table
ax = axes[0]
table_data = np.array([[both_right, a_right_b_wrong],
                        [a_wrong_b_right, both_wrong]])
im = ax.imshow(table_data, cmap='Blues')
ax.set_xticks([0, 1])
ax.set_yticks([0, 1])
ax.set_xticklabels(['B Correct', 'B Wrong'])
ax.set_yticklabels(['A Correct', 'A Wrong'])
ax.set_title('Contingency Table')
for i in range(2):
    for j in range(2):
        color = 'white' if table_data[i, j] > table_data.max() / 2 else 'black'
        ax.text(j, i, str(table_data[i, j]), ha='center', va='center',
                color=color, fontsize=16)

# Difference distribution
ax = axes[1]
vals, counts = np.unique(diffs_per_sample, return_counts=True)
ax.bar(vals, counts, width=0.4, color='#4CAF50', alpha=0.8, edgecolor='white')
ax.set_xlabel('Per-Sample Difference (A correct - B correct)')
ax.set_ylabel('Count')
ax.set_title('Distribution of Per-Sample Differences')
ax.set_xticks([-1, 0, 1])
ax.set_xticklabels(['-1\n(B right, A wrong)', '0\n(Both agree)', '+1\n(A right, B wrong)'])
ax.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

print(f"Both correct: {both_right}, Both wrong: {both_wrong}")
print(f"A right/B wrong: {a_right_b_wrong}, A wrong/B right: {a_wrong_b_right}")
print(f"Only the disagreements matter for statistical tests.")

## 4. McNemar's Test

Specifically designed for comparing two classifiers on the same test set.

Uses only the **discordant** pairs (where models disagree):

| | B correct | B wrong |
|--|-----------|--------|
| **A correct** | a (both right) | b (A right, B wrong) |
| **A wrong** | c (A wrong, B right) | d (both wrong) |

$$\chi^2 = \frac{(b - c)^2}{b + c}$$

H0: b = c (models make the same types of errors).

In [ ]:
def mcnemar_test(y_true, y_pred_a, y_pred_b, correction=True):
    """McNemar's test for comparing two classifiers.
    
    Args:
        correction: if True, use continuity correction (Edwards)
    """
    correct_a = (y_pred_a == y_true)
    correct_b = (y_pred_b == y_true)
    
    # Contingency table
    b = np.sum(correct_a & ~correct_b)   # A right, B wrong
    c = np.sum(~correct_a & correct_b)   # A wrong, B right
    
    if b + c == 0:
        return {'chi2': 0, 'p_value': 1.0, 'b': b, 'c': c}
    
    # Chi-squared statistic
    if correction:
        chi2 = (abs(b - c) - 1) ** 2 / (b + c)  # continuity correction
    else:
        chi2 = (b - c) ** 2 / (b + c)
    
    # p-value from chi2 distribution with 1 df
    # Using the survival function approximation
    # For chi2 with 1 df: p = erfc(sqrt(chi2/2))
    from math import erfc, sqrt
    p_value = erfc(sqrt(chi2 / 2))
    
    return {
        'chi2': chi2,
        'p_value': p_value,
        'b': int(b),
        'c': int(c),
        'discordant': int(b + c),
    }


result_mc = mcnemar_test(y_true, y_pred_a, y_pred_b)
print(f"McNemar's Test:")
print(f"  b (A right, B wrong): {result_mc['b']}")
print(f"  c (A wrong, B right): {result_mc['c']}")
print(f"  Discordant pairs:     {result_mc['discordant']}")
print(f"  Chi-squared:          {result_mc['chi2']:.4f}")
print(f"  p-value:              {result_mc['p_value']:.4f}")
print(f"\n  Significant at alpha=0.05? {'Yes' if result_mc['p_value'] < 0.05 else 'No'}")
print(f"\nMcNemar's focuses only on where models disagree — the most informative samples.")

## 5. Permutation Test

The most general non-parametric test. Under H0 (no difference), the model labels are interchangeable.

Algorithm:
1. Compute observed difference D_obs
2. For each permutation: randomly swap model A and B predictions for each sample
3. Compute permuted difference D_perm
4. p-value = fraction of |D_perm| >= |D_obs|

In [ ]:
def permutation_test(metric_fn, y_true, y_pred_a, y_pred_b,
                      n_permutations=10000, seed=42):
    """Permutation test for comparing two models.
    
    For each sample, randomly assign it to model A or B.
    Computes two-sided p-value.
    """
    rng = np.random.RandomState(seed)
    n = len(y_true)
    
    # Observed difference
    obs_diff = metric_fn(y_true, y_pred_a) - metric_fn(y_true, y_pred_b)
    
    perm_diffs = []
    for _ in range(n_permutations):
        # For each sample, randomly decide whether to swap A and B
        swap = rng.randint(0, 2, size=n).astype(bool)
        
        perm_a = np.where(swap, y_pred_b, y_pred_a)
        perm_b = np.where(swap, y_pred_a, y_pred_b)
        
        perm_diff = metric_fn(y_true, perm_a) - metric_fn(y_true, perm_b)
        perm_diffs.append(perm_diff)
    
    perm_diffs = np.array(perm_diffs)
    
    # Two-sided p-value
    p_value = np.mean(np.abs(perm_diffs) >= np.abs(obs_diff))
    
    return obs_diff, p_value, perm_diffs


obs_diff, p_perm, perm_dists = permutation_test(
    accuracy_metric, y_true, y_pred_a, y_pred_b)

print(f"Permutation Test:")
print(f"  Observed difference: {obs_diff:.4f}")
print(f"  p-value:             {p_perm:.4f}")
print(f"  Significant at alpha=0.05? {'Yes' if p_perm < 0.05 else 'No'}")

In [ ]:
# Visualize permutation distribution
fig, ax = plt.subplots(figsize=(9, 5))

ax.hist(perm_dists, bins=50, alpha=0.7, color='#9E9E9E', edgecolor='white',
        label='Permutation distribution (H0)')
ax.axvline(obs_diff, color='#F44336', linewidth=2, linestyle='-',
           label=f'Observed diff = {obs_diff:.4f}')
ax.axvline(-obs_diff, color='#F44336', linewidth=2, linestyle='--', alpha=0.5)

# Shade rejection region
mask = np.abs(perm_dists) >= np.abs(obs_diff)
if mask.any():
    extreme_vals = perm_dists[mask]

ax.set_xlabel('Accuracy Difference (A - B)')
ax.set_ylabel('Count')
ax.set_title(f'Permutation Test (p = {p_perm:.4f})')
ax.legend()
ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print("Under H0, the difference distribution is centered at 0.")
print(f"p-value = fraction of permutations with |diff| >= |{obs_diff:.4f}|")

## 6. Comparing All Tests on the Same Data

In [ ]:
# Summary comparison
print(f"All tests on the same model comparison (n={n_test}):")
print(f"  Model A accuracy: {acc_a:.4f}")
print(f"  Model B accuracy: {acc_b:.4f}")
print(f"  Difference:       {acc_a - acc_b:.4f}")
print()

# Run all tests
_, _, _, p_boot = bootstrap_paired_difference(accuracy_metric, y_true, y_pred_a, y_pred_b)
t_result = paired_t_test(y_true, y_pred_a, y_pred_b)
mc_result = mcnemar_test(y_true, y_pred_a, y_pred_b)
_, p_perm_val, _ = permutation_test(accuracy_metric, y_true, y_pred_a, y_pred_b)

print(f"{'Test':<25} {'p-value':>10} {'Significant (a=0.05)':>22}")
print("-" * 60)
print(f"{'Bootstrap (paired)':25} {p_boot:>10.4f} {'Yes' if p_boot < 0.05 else 'No':>22}")
print(f"{'Paired t-test':25} {t_result['p_value']:>10.4f} {'Yes' if t_result['p_value'] < 0.05 else 'No':>22}")
print(f"{'McNemar (corrected)':25} {mc_result['p_value']:>10.4f} {'Yes' if mc_result['p_value'] < 0.05 else 'No':>22}")
print(f"{'Permutation test':25} {p_perm_val:>10.4f} {'Yes' if p_perm_val < 0.05 else 'No':>22}")

## 7. Effect of Sample Size on Power

In [ ]:
# How does test set size affect our ability to detect a real difference?
true_acc_a = 0.85
true_acc_b = 0.83
sample_sizes = [50, 100, 200, 500, 1000, 2000, 5000]
n_trials = 200

power_bootstrap = []
power_mcnemar = []

for n in sample_sizes:
    sig_boot = 0
    sig_mcn = 0
    
    for trial in range(n_trials):
        rng = np.random.RandomState(trial * 100 + n)
        yt = rng.binomial(1, 0.5, n)
        noise_a = rng.rand(n)
        noise_b = rng.rand(n)
        ypa = np.where(noise_a < true_acc_a, yt, 1 - yt)
        ypb = np.where(noise_b < true_acc_b, yt, 1 - yt)
        
        # Bootstrap test
        _, _, _, p_b = bootstrap_paired_difference(
            accuracy_metric, yt, ypa, ypb, n_bootstrap=1000, seed=trial)
        if p_b < 0.05:
            sig_boot += 1
        
        # McNemar test
        mc_r = mcnemar_test(yt, ypa, ypb)
        if mc_r['p_value'] < 0.05:
            sig_mcn += 1
    
    power_bootstrap.append(sig_boot / n_trials)
    power_mcnemar.append(sig_mcn / n_trials)

fig, ax = plt.subplots(figsize=(9, 5))
ax.plot(sample_sizes, power_bootstrap, 'o-', color='#2196F3', linewidth=2, label='Bootstrap')
ax.plot(sample_sizes, power_mcnemar, 's-', color='#FF9800', linewidth=2, label='McNemar')
ax.axhline(0.8, color='gray', linestyle='--', alpha=0.5, label='80% power')
ax.axhline(0.05, color='red', linestyle=':', alpha=0.5, label='Type I error rate')
ax.set_xlabel('Test Set Size')
ax.set_ylabel('Power (Fraction of Significant Results)')
ax.set_title(f'Statistical Power vs Sample Size (true diff = {true_acc_a - true_acc_b:.0%})')
ax.legend()
ax.grid(True, alpha=0.3)
ax.set_xscale('log')
plt.tight_layout()
plt.show()

print("Power = probability of detecting a real difference when it exists.")
print("For a 2% accuracy difference, you need ~500-1000 test samples for reliable detection.")

## 8. Multiple Comparisons: Bonferroni Correction

In [ ]:
# When comparing K models, you make K*(K-1)/2 pairwise comparisons.
# Without correction, you inflate the false positive rate.

def bonferroni_correction(p_values, alpha=0.05):
    """Bonferroni correction for multiple comparisons.
    
    Adjusted threshold = alpha / number_of_tests.
    """
    n_tests = len(p_values)
    adjusted_alpha = alpha / n_tests
    adjusted_p = np.minimum(np.array(p_values) * n_tests, 1.0)
    
    return {
        'adjusted_alpha': adjusted_alpha,
        'adjusted_p_values': adjusted_p,
        'significant': adjusted_p < alpha,
    }


# Simulate comparing 5 models (4 real improvements + 1 null)
np.random.seed(42)
n = 1000
y_t = np.random.binomial(1, 0.5, n)

# Baseline model: 80% accuracy
noise = np.random.rand(n)
y_base = np.where(noise < 0.80, y_t, 1 - y_t)

# 4 improved models + 1 no-improvement
models = {
    'Model +3%': 0.83,
    'Model +2%': 0.82,
    'Model +1%': 0.81,
    'Model +0.5%': 0.805,
    'Model +0% (null)': 0.80,
}

p_values = []
model_names = []
for name, acc in models.items():
    noise_m = np.random.rand(n)
    y_m = np.where(noise_m < acc, y_t, 1 - y_t)
    _, _, _, p = bootstrap_paired_difference(
        accuracy_metric, y_t, y_m, y_base, n_bootstrap=5000)
    p_values.append(p)
    model_names.append(name)

correction = bonferroni_correction(p_values)

print(f"{'Model':<20} {'p-value':>10} {'Adjusted p':>12} {'Sig (raw)':>12} {'Sig (Bonf)':>12}")
print("-" * 68)
for i, name in enumerate(model_names):
    print(f"{name:<20} {p_values[i]:>10.4f} {correction['adjusted_p_values'][i]:>12.4f} "
          f"{'Yes' if p_values[i] < 0.05 else 'No':>12} "
          f"{'Yes' if correction['significant'][i] else 'No':>12}")

print(f"\nBonferroni threshold: {correction['adjusted_alpha']:.4f} (= 0.05 / {len(p_values)})")
print(f"Bonferroni is conservative — it controls family-wise error rate.")

## 9. Practical Workflow

In [ ]:
def full_model_comparison(y_true, y_pred_a, y_pred_b, model_a_name='Model A',
                           model_b_name='Model B', n_bootstrap=10000):
    """Complete comparison workflow."""
    acc_a = accuracy_metric(y_true, y_pred_a)
    acc_b = accuracy_metric(y_true, y_pred_b)
    
    print(f"=== Comparing {model_a_name} vs {model_b_name} ===")
    print(f"  {model_a_name}: {acc_a:.4f}")
    print(f"  {model_b_name}: {acc_b:.4f}")
    print(f"  Difference: {acc_a - acc_b:+.4f}")
    print()
    
    # Bootstrap CI
    diff, ci, _, p_boot = bootstrap_paired_difference(
        accuracy_metric, y_true, y_pred_a, y_pred_b, n_bootstrap=n_bootstrap)
    print(f"  Bootstrap 95% CI for difference: [{ci[0]:.4f}, {ci[1]:.4f}]")
    
    # McNemar
    mc = mcnemar_test(y_true, y_pred_a, y_pred_b)
    
    # Permutation
    _, p_perm, _ = permutation_test(accuracy_metric, y_true, y_pred_a, y_pred_b)
    
    print(f"\n  p-values:")
    print(f"    Bootstrap:   {p_boot:.4f}")
    print(f"    McNemar:     {mc['p_value']:.4f}")
    print(f"    Permutation: {p_perm:.4f}")
    
    # Verdict
    any_sig = p_boot < 0.05 or mc['p_value'] < 0.05 or p_perm < 0.05
    all_sig = p_boot < 0.05 and mc['p_value'] < 0.05 and p_perm < 0.05
    
    print(f"\n  Verdict: ", end='')
    if all_sig:
        better = model_a_name if diff > 0 else model_b_name
        print(f"{better} is significantly better (all tests agree).")
    elif any_sig:
        print(f"Mixed evidence — some tests significant, some not. Collect more data.")
    else:
        print(f"No significant difference detected. Need more data or larger effect.")


# Run full comparison
full_model_comparison(y_true, y_pred_a, y_pred_b, 'Model A (85%)', 'Model B (84%)')

print("\n" + "="*60 + "\n")

# Compare with a clearly better model
np.random.seed(99)
noise_c = np.random.rand(n_test)
y_pred_c = np.where(noise_c < 0.92, y_true, 1 - y_true)
full_model_comparison(y_true, y_pred_c, y_pred_b, 'Model C (92%)', 'Model B (84%)')

## Interview Questions & Answers

---

**Q: How to know if improvement is significant?**

1. **Bootstrap confidence interval** on the difference: if the 95% CI excludes 0, the difference is significant at alpha=0.05.
2. **McNemar's test**: for classification, tests if the disagreement pattern is asymmetric. Best for binary outcomes.
3. **Permutation test**: most general. Shuffles model assignments and measures how extreme the observed difference is.

Rule of thumb: report bootstrap CIs (easy to interpret) and one formal test (McNemar for classification, permutation for other metrics).

---

**Q: Why bootstrap over parametric tests?**

1. **No distributional assumptions**: t-test assumes normal differences. For metrics like AUC, F1, BLEU — the distribution is unknown.
2. **Works with any metric**: bootstrap works with any computable metric. Parametric tests often only exist for simple statistics.
3. **Gives a full confidence interval**, not just a p-value.
4. **Paired bootstrap** accounts for the correlation between models evaluated on the same test set.

Downsides: computationally heavier (but trivial with modern hardware), and bootstrap p-values have finite resolution (1/B).

---

**Q: Multiple comparisons — why does it matter?**

If you compare 10 models pairwise (45 comparisons) at alpha=0.05, you expect ~2.25 false positives even if all models are identical. Bonferroni correction: use alpha/K as the threshold. Alternatives: Holm-Bonferroni (less conservative), Benjamini-Hochberg (controls FDR instead of FWER).

---

**Q: How large should the test set be?**

Depends on the expected effect size. For a 1% accuracy difference, you typically need 1000+ test samples. For a 5% difference, 200-500 may suffice. Use a power analysis or bootstrap pilot study to estimate required sample size.

## Quick Reference

```
Bootstrap CI:  Resample test set B times, compute statistic, take percentiles.
Paired t-test: t = mean(d) / (std(d)/sqrt(n)) where d_i = correct_A(i) - correct_B(i)
McNemar:       chi2 = (b-c)^2 / (b+c), only uses discordant pairs
Permutation:   Shuffle A/B labels, compute diff, p = fraction >= observed

When to use what:
  - Bootstrap:    any metric, no assumptions, gives CI + p-value
  - Paired t-test: quick, parametric, assumes normality of differences
  - McNemar:       binary classification, exact test for disagreements
  - Permutation:   exact, any metric, no assumptions

Multiple comparisons: Bonferroni (alpha/K), Holm, or BH correction
```